In [0]:
from pyspark.sql import functions as F


catalog = "workspace"
schema = "default"


# ============================================================================
# 1. FLIGHT KPI
# ============================================================================

flight_df = spark.table(
    f"{catalog}.{schema}.silver_flights"
)

flight_kpi = (
    flight_df
    .agg(
        F.countDistinct("flight_id").alias(
            "total_flights"
        ),
        F.countDistinct("aircraft_id").alias(
            "total_aircraft"
        ),
        F.sum("passengers").alias(
            "total_passengers"
        ),
        F.round(
            F.avg("delay_minutes"), 2
        ).alias(
            "average_delay_minutes"
        ),
        F.sum(
            F.when(
                F.col("status") == "Delayed",
                1
            ).otherwise(0)
        ).alias(
            "delayed_flights"
        )
    )
)


# ============================================================================
# 2. INCIDENT KPI
# ============================================================================

incident_df = spark.table(
    f"{catalog}.{schema}.silver_flight_incidents"
)

incident_kpi = (
    incident_df
    .agg(
        F.countDistinct("incident_id").alias(
            "total_incidents"
        ),
        F.sum(
            F.when(
                F.col("severity") == "High",
                1
            ).otherwise(0)
        ).alias(
            "high_severity_incidents"
        ),
        F.sum(
            F.when(
                F.col("resolution_status") == "Open",
                1
            ).otherwise(0)
        ).alias(
            "open_incidents"
        )
    )
)


# ============================================================================
# 3. MAINTENANCE KPI
# ============================================================================

maintenance_df = spark.table(
    f"{catalog}.{schema}.silver_maintenance"
)

maintenance_kpi = (
    maintenance_df
    .agg(
        F.countDistinct("maintenance_id").alias(
            "total_maintenance_events"
        ),
        F.countDistinct("aircraft_id").alias(
            "maintenance_aircraft"
        ),
        F.round(
            F.sum("cost_usd"), 2
        ).alias(
            "total_maintenance_cost_usd"
        ),
        F.round(
            F.sum("downtime_hours"), 2
        ).alias(
            "total_downtime_hours"
        )
    )
)


# ============================================================================
# 4. SENSOR KPI
# ============================================================================

sensor_df = spark.table(
    f"{catalog}.{schema}.silver_sensor_data"
)

sensor_kpi = (
    sensor_df
    .agg(
        F.count("sensor_id").alias(
            "total_sensor_readings"
        ),
        F.sum(
            F.when(
                F.col("anomaly_status") == "Warning",
                1
            ).otherwise(0)
        ).alias(
            "warning_sensor_readings"
        ),
        F.sum(
            F.when(
                F.col("anomaly_status") == "Critical",
                1
            ).otherwise(0)
        ).alias(
            "critical_sensor_readings"
        )
    )
)


# ============================================================================
# 5. TELEMETRY KPI
# ============================================================================

telemetry_df = spark.table(
    f"{catalog}.{schema}.silver_aerospace_flight_telemetry"
)

telemetry_kpi = (
    telemetry_df
    .agg(
        F.count("timestamp").alias(
            "total_telemetry_records"
        ),
        F.countDistinct("flight_id").alias(
            "telemetry_flights"
        ),
        F.round(
            F.avg("altitude_ft"), 2
        ).alias(
            "average_altitude_ft"
        ),
        F.round(
            F.avg("indicated_airspeed_knots"), 2
        ).alias(
            "average_airspeed_knots"
        ),
        F.sum(
            F.when(
                F.col("display_alert_flag") == 1,
                1
            ).otherwise(0)
        ).alias(
            "total_display_alerts"
        )
    )
)


# ============================================================================
# 6. CREATE FINAL KPI TABLE
# ============================================================================

final_report = (
    flight_kpi
    .crossJoin(incident_kpi)
    .crossJoin(maintenance_kpi)
    .crossJoin(sensor_kpi)
    .crossJoin(telemetry_kpi)
    .withColumn(
        "report_generated_timestamp",
        F.current_timestamp()
    )
)


# ============================================================================
# 7. SAVE FINAL REPORT
# ============================================================================

(
    final_report
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.default.aerospace_final_kpi_report"
    )
)


# ============================================================================
# 8. DISPLAY
# ============================================================================

display(final_report)


print("==============================================")
print("FINAL REPORTING COMPLETED")
print("==============================================")

In [0]:
display(
    spark.table("workspace.default.aerospace_final_kpi_report")
)